In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

# Cargar el archivo de inferencia de ventas 2025
inferencia_df = pd.read_csv("../data/raw/inferencia/ventas_2025_inferencia.csv")

# Mostrar las primeras filas para verificar la carga
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [25]:
# ==========================================
# TRANSFORMACIÓN DE INFERENCIA_DF
# Aplicando las mismas transformaciones que se hicieron con df en el notebook de entrenamiento
# ==========================================

# 1. Variables básicas de fecha
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
inferencia_df['nombre_dia'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week

# 2. Es fin de semana
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5, 6]).astype(int)

# 3. Festivos en España
festivos = holidays.country_holidays('ES', years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos).astype(int)

# 4. Black Friday (último viernes de noviembre)
def es_black_friday(fecha):
    año = fecha.year
    nov = pd.date_range(start=f'{año}-11-01', end=f'{año}-11-30', freq='D')
    fridays = nov[nov.weekday == 4]
    return fecha == fridays[-1]

inferencia_df['es_BlackFriday'] = inferencia_df['fecha'].apply(es_black_friday).astype(int)

# 5. Cyber Monday (primer lunes después de Black Friday)
def es_cyber_monday(fecha):
    año = fecha.year
    nov = pd.date_range(start=f'{año}-11-01', end=f'{año}-11-30', freq='D')
    fridays = nov[nov.weekday == 4]
    bf = fridays[-1]
    cm = bf + pd.Timedelta(days=(7 - bf.weekday()))
    return fecha == cm

inferencia_df['es_CyberMonday'] = inferencia_df['fecha'].apply(es_cyber_monday).astype(int)

# 6. Variable: es_primer_dia_mes
inferencia_df['es_primer_dia_mes'] = (inferencia_df['dia_mes'] == 1).astype(int)

# 7. Variable: es_ultimo_dia_mes
inferencia_df['es_ultimo_dia_mes'] = (inferencia_df['fecha'] == inferencia_df['fecha'] + pd.offsets.MonthEnd(0)).astype(int)

# 8. Variable: trimestre
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter

# 9. Variable: mitad_mes (1=primera quincena, 2=segunda quincena)
inferencia_df['mitad_mes'] = np.where(inferencia_df['dia_mes'] <= 15, 1, 2)

# 10. Variable: es_inicio_semana (lunes)
inferencia_df['es_inicio_semana'] = (inferencia_df['dia_semana'] == 0).astype(int)

# 11. Variable: es_fin_semana_dia (domingo)
inferencia_df['es_fin_semana_dia'] = (inferencia_df['dia_semana'] == 6).astype(int)

# 12. Variable: es_laborable (no festivo y no fin de semana)
inferencia_df['es_laborable'] = ((inferencia_df['es_festivo'] == 0) & (inferencia_df['es_fin_semana'] == 0)).astype(int)

print("Variables temporales y de calendario creadas ✓")
inferencia_df.head()

C:\Users\alexa\AppData\Local\Temp\ipykernel_32276\1516251434.py:20: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(festivos).astype(int)


Variables temporales y de calendario creadas ✓


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,es_festivo,es_BlackFriday,es_CyberMonday,es_primer_dia_mes,es_ultimo_dia_mes,trimestre,mitad_mes,es_inicio_semana,es_fin_semana_dia,es_laborable
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,0,0,0,0,0,4,2,0,0,0
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,0,0,0,0,0,4,2,0,0,0
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,0,0,0,0,0,4,2,0,0,0
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,0,0,0,0,0,4,2,0,0,0
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,0,0,0,0,0,4,2,0,0,0


In [26]:
# ==========================================
# VARIABLES DE LAG Y MEDIA MÓVIL
# ==========================================

# IMPORTANTE: Para inferencia, no tenemos datos históricos de unidades_vendidas
# Por lo tanto, estas variables tendrán valores NaN
# El modelo debe ser capaz de hacer predicciones sin estas variables o usaremos valores por defecto

# Crear lags de 1 a 7 días (todos con NaN para inferencia)
for lag in range(1, 8):
    inferencia_df[f'unidades_vendidas_lag{lag}'] = np.nan

# Media móvil de 7 días (con NaN para inferencia)
inferencia_df['unidades_vendidas_mm7'] = np.nan

print("Variables de lag y media móvil creadas (NaN para inferencia) ✓")
inferencia_df.head()

Variables de lag y media móvil creadas (NaN para inferencia) ✓


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,es_fin_semana_dia,es_laborable,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# ==========================================
# VARIABLES DE PRECIO
# ==========================================

# Variable de descuento porcentaje
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base'] * 100)

# Calcular el precio promedio de la competencia
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)

# Calcular el ratio de nuestro precio respecto a la competencia
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

# Eliminar las columnas de los competidores
inferencia_df = inferencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'])

print("Variables de precio creadas ✓")
inferencia_df.head()

Variables de precio creadas ✓


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7,descuento_porcentaje,precio_competencia,ratio_precio
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.626087,102.573333,1.102918
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.103704,121.506667,1.167755
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.929412,81.453333,1.053241
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.586667,72.330000,1.053367
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.371429,33.190000,1.068997


In [28]:
# ==========================================
# ONE-HOT ENCODING
# ==========================================

# Crear copias de las variables con sufijo _h
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# One Hot Encoding sobre las nuevas variables
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], prefix=['nombre_h', 'categoria_h', 'subcategoria_h'])

print("One-Hot Encoding aplicado ✓")
print(f"Shape del dataframe: {inferencia_df.shape}")
inferencia_df.head()

One-Hot Encoding aplicado ✓
Shape del dataframe: (888, 82)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,False,False,False,False,False,False,False,False,True,False
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,False,False,False,False,False,False,False,False,True,False
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,False,False,False,False,False,False,False,False,True,False
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,False,False,False,False,False,False,False,False,True,False
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,False,False,False,False,False,False,False,True,False,False


In [29]:
# ==========================================
# FILTRAR SOLO NOVIEMBRE (eliminar octubre)
# ==========================================

print(f"Registros antes de filtrar: {len(inferencia_df)}")
print(f"Meses únicos antes: {sorted(inferencia_df['mes'].unique())}")

# Filtrar solo noviembre (mes == 11)
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()

print(f"Registros después de filtrar: {len(inferencia_df)}")
print(f"Meses únicos después: {sorted(inferencia_df['mes'].unique())}")
print("Filtrado completado ✓")

Registros antes de filtrar: 888
Meses únicos antes: [np.int32(10), np.int32(11)]
Registros después de filtrar: 720
Meses únicos después: [np.int32(11)]
Filtrado completado ✓


In [30]:
# ==========================================
# GUARDAR DATAFRAME TRANSFORMADO
# ==========================================

# Guardar el dataframe transformado en data/processed
inferencia_df.to_csv("../data/processed/inferencia_df_transformado.csv", index=False)

print(f"✓ Dataframe guardado exitosamente en data/processed/inferencia_df_transformado.csv")
print(f"✓ Shape final: {inferencia_df.shape}")
print(f"✓ Columnas totales: {len(inferencia_df.columns)}")
print(f"\nPrimeras columnas:")
print(inferencia_df.columns.tolist()[:20])

✓ Dataframe guardado exitosamente en data/processed/inferencia_df_transformado.csv
✓ Shape final: (720, 82)
✓ Columnas totales: 82

Primeras columnas:
['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'año', 'mes', 'dia_mes', 'dia_semana', 'nombre_dia', 'semana_año', 'es_fin_semana', 'es_festivo', 'es_BlackFriday', 'es_CyberMonday']


In [33]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'mes', 'dia_mes', 'dia_semana', 'nombre_dia',
       'semana_año', 'es_fin_semana', 'es_festivo', 'es_BlackFriday',
       'es_CyberMonday', 'es_primer_dia_mes', 'es_ultimo_dia_mes', 'trimestre',
       'mitad_mes', 'es_inicio_semana', 'es_fin_semana_dia', 'es_laborable',
       'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7',
       'descuento_porcentaje', 'precio_competencia', 'ratio_precio',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_